In [1]:
import pandas as pd 
import numpy as np

In [2]:
MR = pd.read_csv("movies.csv")

In [3]:
MR.head()

,movieId,title,genres
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
1,2,Jumanji (1995),Adventure|Children|Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance
4,5,Father of the Bride Part II (1995),Comedy


In [4]:
import re 
def clean_title(title):
    return re.sub("[^a-zA-Z0-9]", " ", title)
    

In [5]:
MR["clean_title"] = MR["title"].apply(clean_title)


In [6]:
MR

,movieId,title,genres,clean_title
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy,Toy Story 1995
1,2,Jumanji (1995),Adventure|Children|Fantasy,Jumanji 1995
2,3,Grumpier Old Men (1995),Comedy|Romance,Grumpier Old Men 1995
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance,Waiting to Exhale 1995
4,5,Father of the Bride Part II (1995),Comedy,Father of the Bride Part II 1995
...,...,...,...,...
62418,209157,We (2018),Drama,We 2018
62419,209159,Window of the Soul (2001),Documentary,Window of the Soul 2001
62420,209163,Bad Poems (2018),Comedy|Drama,Bad Poems 2018
62421,209169,A Girl Thing (2001),(no genres listed),A Girl Thing 2001


Cleaning movie title with regex 

In [7]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer(ngram_range=(1,2)) #cleaning our dataset from any unwanted signs 
tfidf = vectorizer.fit_transform(MR["clean_title"])

Creating a function that takes the name of the wanted movie and compare it with the ones we have in the file and return the most similar ones 

In [18]:

from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

def search(title):
    title = clean_title(title)
    query_vec = vectorizer.transform([title])
    similarity = cosine_similarity(query_vec, tfidf).flatten()

    indices = np.argpartition(similarity, -5)[-5:]
    indices = indices[np.argsort(-similarity[indices])]

    results = MR.iloc[indices]
    return results


Building an intractive search box in which the use can enter the movie he wants similarities with

In [ ]:

import ipywidgets as widgets
from IPython.display import display

movie_input = widgets.Text(
    value="",
    description="Movie Title:",
    placeholder="Example: Toy Story",
    disabled=False
)
movie_list = widgets.Output()
def on_type(data):
    with movie_list:
        movie_list.clear_output()
        title = data["new"]
        if len(title) > 5:
            display(search(title))

movie_input.observe(on_type, names ="value")

display(movie_input, movie_list)


Text(value='', description='Movie Title:', placeholder='Example: Toy Story')

Output()

Reading in maovie ratings data 

In [19]:
ratings = pd.read_csv("ratings.csv")

In [20]:
ratings.head()

,userId,movieId,rating,timestamp
0,1,296,5.0,1147880044
1,1,306,3.5,1147868817
2,1,307,5.0,1147868828
3,1,665,5.0,1147878820
4,1,899,3.5,1147868510


Fiding uses who likes the same movie 

In [22]:
movieId = 1

In [24]:
similar_users = ratings[(ratings["movieId"] == movieId) &(ratings["rating"] >= 5)]["userId"].unique()

In [25]:
similar_users 

array([    36,     75,     86, ..., 162518, 162519, 162530],
      shape=(13506,))

In [58]:
similar_users_recs = ratings[(ratings["userId"].isin(similar_users)) & (ratings["rating"]>4)]["movieId"]

In [59]:
similar_users_recs

5101           1
5105          34
5111         110
5114         150
5127         260
            ... 
24998388    3706
24998389    3735
24998391    3763
24998392    4187
24998393    4321
Name: movieId, Length: 912084, dtype: int64

In [66]:
similar_users_recs = similar_users_recs.value_counts() / len(similar_users)
similar_users_recs = similar_users_recs[similar_users_recs > .1]

In [67]:
similar_users_recs

movieId
1       1.000000
318     0.414556
260     0.404561
356     0.347253
296     0.342663
          ...   
1259    0.102991
7361    0.101881
1206    0.101362
1307    0.101066
1208    0.100918
Name: count, Length: 92, dtype: float64

In [68]:
all_users = ratings[(ratings["movieId"].isin(similar_users_recs.index)) & (ratings["rating"] > 4)]

In [69]:
all_users

,userId,movieId,rating,timestamp
0,1,296,5.0,1147880044
29,1,4973,4.5,1147869080
48,1,7361,5.0,1147880055
72,2,110,5.0,1141416589
76,2,260,5.0,1141417172
...,...,...,...,...
25000055,162541,4973,4.5,1240950790
25000057,162541,4993,5.0,1240952610
25000065,162541,5952,5.0,1240952617
25000078,162541,7153,5.0,1240952613


In [70]:
all_users_recs = all_users["movieId"].value_counts() / len(all_users["userId"].unique())

In [71]:
all_users_recs

movieId
318      0.345282
296      0.287220
2571     0.246217
356      0.237370
593      0.227930
           ...   
1387     0.047886
1307     0.046195
745      0.037362
78499    0.035445
2355     0.025316
Name: count, Length: 92, dtype: float64

creating a recommendation score 

In [72]:
rec_precentage = pd.concat([similar_users_recs, all_users_recs],axis=1)
rec_precentage.columns = ["similar", "all"]

In [73]:
rec_precentage

,similar,all
movieId,,
1,1.000000,0.125844
318,0.414556,0.345282
260,0.404561,0.224195
356,0.347253,0.237370
296,0.342663,0.287220
...,...,...
1259,0.102991,0.049349
7361,0.101881,0.105172
1206,0.101362,0.087500


In [74]:
rec_precentage["score"] = rec_precentage["similar"] / rec_precentage["all"]

In [75]:
rec_precentage = rec_precentage.sort_values("score", ascending=False)


In [76]:
rec_precentage

,similar,all,score
movieId,,,
1,1.000000,0.125844,7.946323
3114,0.295498,0.054186,5.453383
2355,0.124685,0.025316,4.925186
78499,0.138161,0.035445,3.897906
588,0.233674,0.068117,3.430480
...,...,...,...
58559,0.160743,0.147779,1.087725
79132,0.129424,0.132559,0.976349
7361,0.101881,0.105172,0.968704


In [78]:
rec_precentage.head(11).merge(MR, on="movieId", how="left")

,movieId,similar,all,score,title,genres,clean_title
0,1,1.000000,0.125844,7.946323,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy,Toy Story 1995
1,3114,0.295498,0.054186,5.453383,Toy Story 2 (1999),Adventure|Animation|Children|Comedy|Fantasy,Toy Story 2 1999
2,2355,0.124685,0.025316,4.925186,"Bug's Life, A (1998)",Adventure|Animation|Children|Comedy,Bug s Life A 1998
3,78499,0.138161,0.035445,3.897906,Toy Story 3 (2010),Adventure|Animation|Children|Comedy|Fantasy|IMAX,Toy Story 3 2010
4,588,0.233674,0.068117,3.430480,Aladdin (1992),Adventure|Animation|Children|Comedy|Musical,Aladdin 1992
5,595,0.198949,0.060514,3.287671,Beauty and the Beast (1991),Animation|Children|Fantasy|Musical|Romance|IMAX,Beauty and the Beast 1991
6,34,0.158226,0.052696,3.002602,Babe (1995),Children|Drama,Babe 1995
7,4886,0.210647,0.071444,2.948410,"Monsters, Inc. (2001)",Adventure|Animation|Children|Comedy|Fantasy,Monsters Inc 2001
8,1073,0.143418,0.049202,2.914882,Willy Wonka & the Chocolate Factory (1971),Children|Comedy|Fantasy|Musical,Willy Wonka the Chocolate Factory 1971
9,745,0.108322,0.037362,2.899227,Wallace & Gromit: A Close Shave (1995),Animation|Children|Comedy,Wallace Gromit A Close Shave 1995


Building a recommendation function 

In [ ]:
def find_similar_movies(movie_id):
    similar_users = ratings[(ratings["movieId"] == movie_id) & (ratings["rating"] >= 5)]["userId"].unique()
    similar_users_recs = ratings[(ratings["userId"].isin(similar_users)) & (ratings["rating"] > 4)]["movieId"]
    similar_users_recs = similar_users_recs.value_counts() / len(similar_users)
    similar_users_recs = similar_users_recs[similar_users_recs > .1]
    
    all_users = ratings[(ratings["movieId"].isin(similar_users_recs.index)) & (ratings["rating"] > 4)]
    all_users_recs = all_users["movieId"].value_counts() / len(all_users["userId"].unique())
    
    rec_precentage = pd.concat([similar_users_recs, all_users_recs], axis=1)
    rec_precentage.columns = ["similar", "all"]
    rec_precentage["score"] = rec_precentage["similar"] / rec_precentage["all"]
    rec_precentage = rec_precentage.sort_values("score", ascending=False)
    
    return rec_precentage.head(11).merge(MR, left_index=True, right_on="movieId")[["movieId", "title", "score"]]

creating an intractive recommendation widge 

In [82]:
movie_input_name = widgets.Text(
    value="",
    description="Movie Title:",
    placeholder="Example: Toy Story",   
    disabled=False
)
recommendation_list = widgets.Output()
def on_type(data):
    with recommendation_list:
        recommendation_list.clear_output()
        title = data["new"]
        if len(title) > 5:
            search_results = search(title)
            if not search_results.empty:
                movie_id = search_results.iloc[0]["movieId"]
                recommendations = find_similar_movies(movie_id)
                display(recommendations)

movie_input_name.observe(on_type, names="value")
display(movie_input_name, recommendation_list)

Text(value='', description='Movie Title:', placeholder='Example: Toy Story')

Output()

In [83]:
import joblib

joblib.dump(
    {
        "movies": MR,
        "vectorizer": vectorizer,
        "tfidf": tfidf,
        "ratings": ratings[ratings["rating"] >= 4.5][["userId", "movieId", "rating"]].reset_index(drop=True),
    },
    "model.joblib",
    compress=3,
)
print("Saved model.joblib")

Saved model.joblib
